<a href="https://colab.research.google.com/github/kunitskiialex/Flight_Delay_Analysis/blob/main/%D0%90_B_TESTS_PYTHON.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A/B Testing Template in Python

This notebook provides a structured template for conducting A/B tests using Python. It covers data generation, statistical tests (T-test, Chi-squared, Permutation test), and various visualizations to analyze results. Adapt this template with your own data and adjust parameters as needed.

Цей ноутбук є структурованим шаблоном для проведення A/B тестів за допомогою Python. Він охоплює генерацію даних, статистичні тести (T-тест, Chi-squared, Permutation test) та різні візуалізації для аналізу результатів. Адаптуйте цей шаблон під свої дані та налаштуйте параметри за потребою.

## Summary of A/B Test Results

*Use this section to summarize the key findings and conclusions from your A/B tests. Based on the statistical tests and visualizations, state whether there is a significant difference between groups and what the practical implications are.*

## Підсумок результатів A/B тестування

*Використовуйте цей розділ для узагальнення ключових висновків та результатів ваших A/B тестів. На основі статистичних тестів та візуалізацій вкажіть, чи існує значна різниця між групами та які практичні наслідки це має.*

In [ ]:
import pandas as pd

test_data = pd.DataFrame(data={'test_group': ['a']*12673 + ['b']*12145,
                               'conversion': [1]*422 + [0]*(12673-422) + [1]*488 + [0]*(12145-488)})

# Перевіримо, чи все коректно згенерували:
test_data.groupby('test_group').describe()


## Statistical Tests: T-Test

## Статистичні тести: T-Тест

In [ ]:
from scipy import stats

alpha = 0.05

statistic, pvalue = stats.ttest_ind(test_data[test_data['test_group'] == 'a']['conversion'],
                                    test_data[test_data['test_group'] == 'b']['conversion'],
                                    alternative='less')
                                    #alternative — визначає альтернативну гіпотезу.
                                    #less — середнє вибірки a менше за середнє вибірки b;
                                    #greater — середнє вибірки a більше за середнє вибірки b;
                                    #two-sided — середні вибірок відрізняються.
print(f't-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 2)}')

if pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')


## Statistical Tests: Chi-squared Test

## Статистичні тести: Критерій хі-квадрат

In [ ]:
from scipy import stats

alpha = 0.05

observed = pd.crosstab(test_data['test_group'].values, test_data['conversion'].values)
statistic, pvalue, dof, expected_values = stats.chi2_contingency(observed)

print(f't-statistic: {round(statistic, 2)}, p-value: {round(pvalue, 2)}')

if pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')


## Statistical Tests: Permutation Test

## Статистичні тести: Перестановочний тест

In [ ]:
from scipy import stats

def statistic(x, y):
    return stats.ttest_ind(x, y).statistic

alpha = 0.05

x = test_data[test_data['test_group'] == 'a']['conversion']
y = test_data[test_data['test_group'] == 'b']['conversion']

results = stats.permutation_test((x, y), statistic, n_resamples=10000)

print(f'statistic: {round(results.statistic, 2)}, p-value: {round(results.pvalue, 2)}')

if results.pvalue < alpha:
    print('The difference is statistically significant, Null Hypothesis is rejected.')
else:
    print('The difference is insignificant, Null Hypothesis cannot rejected.')


## Visualizations: Group Differences and Confidence Intervals

## Візуалізації: Різниця між групами та довірчі інтервали

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x=test_data['test_group'],
            y=test_data['conversion'],
            errorbar=('ci', 95)) # Confidence Intervals

plt.title('A/B Test Results')
plt.xlabel('Group')
plt.ylabel('Mean')

plt.show()


### Visualizations: Mean Comparison with Standard Error

### Візуалізації: Порівняння середнього зі стандартною похибкою

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.barplot(x=test_data['test_group'],
            y=test_data['conversion'],
            errorbar=('se'))

plt.title('Mean Comparison with Standart Error')
plt.xlabel('Group')
plt.ylabel('Mean')

plt.show()


## Visualizations: Distribution Comparison (KDE Plot)

## Візуалізації: Порівняння розподілу (KDE графік)

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.kdeplot(stats.norm.rvs(size=1000))
sns.kdeplot(stats.norm.rvs(size=1000))

plt.title('Distribution of A/B Groups')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.legend(['A', 'B'])
plt.show()


Цей графік показує оцінку щільності ядра (KDE) для розподілу конверсій у групі A та B. Він допомагає візуально порівняти форми розподілів і зрозуміти, чи відрізняються вони.

In [ ]:
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.histplot(stats.norm.rvs(size=1000))
sns.histplot(stats.norm.rvs(size=1000))

plt.title('Histogram of A/B Groups')
plt.xlabel('Value')
plt.ylabel('Frequency')

plt.legend(['A', 'B'])
plt.show()


Цей графік показує гістограми для розподілу конверсій у групі A та B. Він надає схожу інформацію, що і KDE графік, але у вигляді стовпчикової діаграми частот.

## Visualizations: Cumulative Conversion Rate Over Time

## Візуалізації: Зміна кумулятивної конверсії з часом

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Перемішуємо дані, оскільки зараз вони упорядковані за значенням конверсії
# Використовуючи реальні дані - ТРЕБА сортувати за датою та часом
test_data = test_data.sample(frac=1).reset_index(drop=True)

# Рахуємо кумулятивне середнє - це і є зміна конверсії з плином часу
cumulative_metric_a = test_data[test_data['test_group'] == 'a']['conversion'].expanding().mean().reset_index(drop=True)
cumulative_metric_b = test_data[test_data['test_group'] == 'b']['conversion'].expanding().mean().reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.plot(cumulative_metric_a, label='A')
plt.plot(cumulative_metric_b, label='B')

plt.title('Cumulative Сonversion Rate Comparison')
plt.xlabel('Time')
plt.ylabel('Cumulative Сonversion Rate')

plt.legend()
plt.show()
